# Fig. 4b–d, Supplementary Fig. 6 and Fig. 4e–h

The run-length section at the end (continuous vs spaced trials, ITI = 0 vs 2) also produces Fig. 4e–h.

# Do reward-seeking rules switch between discrete explore/exploit states?

Fits a 2-state Bernoulli HMM to each rule's rewarded-arm-choice sequence at every swept
parameter cell, classifying each as **DYN** (discrete states), **FIX** (seed-multistable
look-alike), or **ns** (not significant / graded). Tested: PC, HR, COV, LC, HZ (BCM, OJA
never reward-seek enough to qualify).

**Result:** all 5 rules have a cell whose two states recur in every one of 20 seeds (chosen
by max-min switch count across seeds). Exploit ≈0.73–1.0; explore ranges ≈0.0 (COV, HR) to
≈0.44 (PC); LC has the weakest separation (p0=0.42, p1=0.73).


In [ ]:
import os
os.environ.setdefault('JAX_PLATFORM_NAME', 'cpu')
import zipfile, warnings
warnings.filterwarnings('ignore')
import logging
for _n in ('matplotlib.font_manager',):
    logging.getLogger(_n).setLevel(logging.ERROR)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['svg.fonttype'] = 'none'

def nature_ax(ax, xlabel='', ylabel='', nticks=4, nx=None, ny=None):
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'): ax.spines[sp].set_linewidth(0.25)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nx or nticks))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(ny or nticks))
    ax.tick_params(labelsize=7, width=0.25, direction='in')
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)

def side_legend(ax, **kw):
    ax.legend(fontsize=7, frameon=False, loc='upper left', bbox_to_anchor=(1.02, 1.0), **kw)

FOLDER = '../../data/det/'
CACHE_DIR = FOLDER + 'cache/'
SIZE = 'small'
os.makedirs('./svgs', exist_ok=True)   # every plt.savefig below writes here
RULES = ['BCM','OJA','LC','PC','COV','HZ','HR']
RULE_ORDER = ['LC','PC','COV','HZ','HR']


## Data and method

`aggregated/det_{RULE}_small.npz['x']`: boolean, True where the trial chose the
deterministically-rewarded arm. Grid: 6 `eta` x 15x15 log-spaced `(alpha, beta)`, plus 4
`tau` for rules with a third timescale (COV, BCM, LC, HZ); `T=50000`, 20 seeds, 7 rules.
Tau-rule arrays are streamed one `eta` slice at a time (full array ~5.4GB).

At each cell, 20 seeds are pooled and a 2-state Bernoulli HMM is fit by scaled Baum-Welch
EM (state 1 = higher-`p` "exploit"), compared to a 1-state null via `2*deltaLL`.


In [ ]:
import numpy.lib.format as fmt

def _read_header(fp):
    """numpy>=2.0 removed _read_array_header; use the public per-version readers."""
    version = fmt.read_magic(fp)
    reader = fmt.read_array_header_1_0 if version == (1, 0) else fmt.read_array_header_2_0
    return reader(fp)

def npz_leading_slices(path, key='x', indices=None):
    """Yield (i, arr) for axis-0 slices of `key`, streaming the DEFLATE stream without
    materializing the full array. indices: axis-0 indices to keep (others are skipped)."""
    zf = zipfile.ZipFile(path)
    want = None if indices is None else set(int(i) for i in indices)
    with zf.open(f'{key}.npy') as fp:
        shape, fortran, dtype = _read_header(fp)
        assert not fortran
        n0 = shape[0]
        slice_shape = shape[1:]
        slice_nbytes = int(np.prod(slice_shape)) * dtype.itemsize
        for i in range(n0):
            if want is not None and i not in want:
                remaining = slice_nbytes
                while remaining > 0:
                    chunk = fp.read(min(remaining, 1 << 20))
                    if not chunk:
                        raise EOFError
                    remaining -= len(chunk)
                continue
            buf = fp.read(slice_nbytes)
            assert len(buf) == slice_nbytes
            arr = np.frombuffer(buf, dtype=dtype).reshape(slice_shape)
            yield i, arr
            if want is not None and i == max(want):
                break

def npz_shape(path, key='x'):
    zf = zipfile.ZipFile(path)
    with zf.open(f'{key}.npy') as fp:
        shape, fortran, dtype = _read_header(fp)
        return shape, dtype


In [ ]:
import jax
import jax.numpy as jnp
jax.config.update('jax_enable_x64', True)
from functools import partial

def _emit(obs_t, p):
    return jnp.where(obs_t[..., None] == 1, p[:, None, :], 1 - p[:, None, :])

def init_params(obs):
    """p0=25th pct of per-seed means, p1=75th pct; A=diagonal-dominant prior; pi=uniform."""
    C = obs.shape[1]
    seed_means = obs.mean(axis=0)
    lo = np.quantile(seed_means, 0.25, axis=1); hi = np.quantile(seed_means, 0.75, axis=1)
    p = np.stack([np.clip(lo, 0.05, 0.95), np.clip(hi, 0.05, 0.95)], axis=1)
    p = np.sort(p, axis=1); p[:, 1] = np.maximum(p[:, 1], p[:, 0] + 0.05)
    A = np.tile(np.array([[0.98, 0.02], [0.02, 0.98]]), (C, 1, 1))
    pi = np.full((C, 2), 0.5)
    return pi, A, p

def relabel_state1_high(pi, A, p):
    """Relabel so state 1 = higher emission prob ("exploit")."""
    pi = pi.copy(); A = A.copy(); p = p.copy()
    swap = p[:, 0] > p[:, 1]
    if swap.any():
        p[swap] = p[swap][:, ::-1]
        pi[swap] = pi[swap][:, ::-1]
        A[swap] = A[swap][:, ::-1, :][:, :, ::-1]
    return pi, A, p

@partial(jax.jit, static_argnums=(1,))
def fit_hmm2(obs, max_iter, pi0, A0, p0, tol=1e-6):
    """Batched 2-state Bernoulli HMM, scaled Baum-Welch EM across C independent groups
    sharing one (pi,A,p) per cell. obs: (T,C,S) 0/1. Runs to convergence (relative change
    in mean log-likelihood < tol) via jax.lax.while_loop around a scan-based
    forward/backward pass. Returns pi,A,p, per-cell loglik, and iterations used."""
    T, C, S = obs.shape

    def em_step(pi, A, p):
        b0 = _emit(obs[0], p)
        a0 = pi[:, None, :] * b0
        c0 = a0.sum(-1); a0 = a0 / c0[..., None]

        def fwd_step(alpha_prev, obs_t):
            bt = _emit(obs_t, p)
            pred = jnp.einsum('csj,cjk->csk', alpha_prev, A)
            a = pred * bt
            ct = a.sum(-1); a = a / ct[..., None]
            return a, (a, ct)

        _, (alphas_rest, cscale_rest) = jax.lax.scan(fwd_step, a0, obs[1:])
        alpha = jnp.concatenate([a0[None], alphas_rest], axis=0)
        cscale = jnp.concatenate([c0[None], cscale_rest], axis=0)
        loglik_total = jnp.log(cscale).sum(axis=(0, 2))

        beta_T = jnp.ones((C, S, 2))
        gT = alpha[-1] * beta_T
        init_sum_gamma = gT.sum(axis=1)
        init_sum_obs_gamma = (gT * obs[-1][..., None]).sum(axis=1)
        init_sum_xi = jnp.zeros((C, 2, 2))

        def bwd_step(carry, xs):
            beta, sum_gamma, sum_obs_gamma, sum_xi = carry
            obs_tp1, alpha_t, cscale_tp1, obs_t = xs
            b_tp1 = _emit(obs_tp1, p)
            tmp = b_tp1 * beta
            beta_new = jnp.einsum('cjk,csk->csj', A, tmp) / cscale_tp1[..., None]
            gamma_t = alpha_t * beta_new
            gamma_t = gamma_t / gamma_t.sum(-1, keepdims=True)
            xi_t = alpha_t[:, :, :, None] * A[:, None, :, :] * tmp[:, :, None, :] / cscale_tp1[:, :, None, None]
            sum_gamma = sum_gamma + gamma_t.sum(axis=1)
            sum_obs_gamma = sum_obs_gamma + (gamma_t * obs_t[..., None]).sum(axis=1)
            sum_xi = sum_xi + xi_t.sum(axis=1)
            return (beta_new, sum_gamma, sum_obs_gamma, sum_xi), gamma_t

        obs_tp1_seq = obs[1:][::-1]
        alpha_t_seq = alpha[:-1][::-1]
        cscale_tp1_seq = cscale[1:][::-1]
        obs_t_seq = obs[:-1][::-1]

        init_carry = (beta_T, init_sum_gamma, init_sum_obs_gamma, init_sum_xi)
        (_, sum_gamma, sum_obs_gamma, sum_xi), gammas = jax.lax.scan(
            bwd_step, init_carry, (obs_tp1_seq, alpha_t_seq, cscale_tp1_seq, obs_t_seq))
        gamma1 = gammas[-1]

        pi_new = gamma1.mean(axis=1); pi_new = pi_new / pi_new.sum(-1, keepdims=True)
        A_new = sum_xi / jnp.clip(sum_xi.sum(axis=2, keepdims=True), 1e-12, None)
        p_new = jnp.clip(sum_obs_gamma / jnp.clip(sum_gamma, 1e-12, None), 1e-4, 1 - 1e-4)
        return pi_new, A_new, p_new, loglik_total

    def cond_fn(state):
        it, pi, A, p, loglik, ll_prev, converged = state
        return jnp.logical_and(it < max_iter, jnp.logical_not(converged))

    def body_fn(state):
        it, pi, A, p, loglik, ll_prev, converged = state
        pi_new, A_new, p_new, loglik_new = em_step(pi, A, p)
        ll_mean = loglik_new.mean()
        converged_new = jnp.logical_and(it > 0, jnp.abs(ll_mean - ll_prev) < tol * jnp.maximum(1.0, jnp.abs(ll_prev)))
        return it + 1, pi_new, A_new, p_new, loglik_new, ll_mean, converged_new

    init_state = (0, pi0, A0, p0, jnp.zeros(C), -jnp.inf, jnp.array(False))
    n_iter_run, pi_f, A_f, p_f, loglik_f, _, _ = jax.lax.while_loop(cond_fn, body_fn, init_state)
    return pi_f, A_f, p_f, loglik_f, n_iter_run


def null_loglik(obs):
    """1-state (iid Bernoulli) log-likelihood per cell, obs (T,C,S)."""
    T, C, S = obs.shape
    pbar = obs.mean(axis=(0, 2))
    x = obs.sum(axis=(0, 2))
    n = T * S
    pbar = np.clip(pbar, 1e-6, 1 - 1e-6)
    return x * np.log(pbar) + (n - x) * np.log(1 - pbar)


@jax.jit
def viterbi(obs, pi, A, p):
    """Batched Viterbi. obs:(T,C,S) 0/1. Returns path (T,C,S) int8 of most-likely states
    (0=explore, 1=exploit, per fit_hmm2's p-sorted convention)."""
    T, C, S = obs.shape
    logA = jnp.log(jnp.clip(A, 1e-12, 1))
    logpi = jnp.log(jnp.clip(pi, 1e-12, 1))

    def logb(obs_t):
        pp = jnp.clip(p, 1e-12, 1 - 1e-12)
        return jnp.where(obs_t[..., None] == 1, jnp.log(pp)[:, None, :], jnp.log(1 - pp)[:, None, :])

    delta0 = logpi[:, None, :] + logb(obs[0])

    def fwd_step(delta_prev, obs_t):
        cand = delta_prev[:, :, :, None] + logA[:, None, :, :]  # (C,S,2j,2k)
        psi_t = jnp.argmax(cand, axis=2)
        delta_t = cand.max(axis=2) + logb(obs_t)
        return delta_t, psi_t

    delta_T, psis = jax.lax.scan(fwd_step, delta0, obs[1:])
    path_T = jnp.argmax(delta_T, axis=-1)

    ci = jnp.arange(C)[:, None]; si = jnp.arange(S)[None, :]

    def bwd_step(path_next, psi_t):
        path_t = psi_t[ci, si, path_next]
        return path_t, path_t

    _, path_rest = jax.lax.scan(bwd_step, path_T, psis, reverse=True)
    return jnp.concatenate([path_rest, path_T[None]], axis=0)


def dwell_stats(path):
    """Per (c,s) run-length stats of the Viterbi path. path:(T,C,S) int8. Returns
    n_switches (C,S), mean_dwell_state1 (C,), mean_dwell_state0 (C,), pooled across seeds."""
    T, C, S = path.shape
    n_switches = (np.diff(path, axis=0) != 0).sum(axis=0)
    dwell0_all = [[] for _ in range(C)]
    dwell1_all = [[] for _ in range(C)]
    for c in range(C):
        for s in range(S):
            x = path[:, c, s]
            change = np.where(np.diff(x) != 0)[0] + 1
            bounds = np.concatenate([[0], change, [T]])
            lens = np.diff(bounds)
            states = x[bounds[:-1]]
            dwell0_all[c].extend(lens[states == 0].tolist())
            dwell1_all[c].extend(lens[states == 1].tolist())
    mean_dwell0 = np.array([np.mean(d) if d else np.nan for d in dwell0_all])
    mean_dwell1 = np.array([np.mean(d) if d else np.nan for d in dwell1_all])
    return dict(n_switches=n_switches, mean_dwell0=mean_dwell0, mean_dwell1=mean_dwell1)


## Per-cell HMM fits

Every reward-seeking `(alpha,beta)` cell is fit (not a subsample): per rule, the 3 `eta`
(best `tau` for tau-rules) with the highest fraction of cells clearing `reward > 0.7`.
Rules with no qualifying `eta` are excluded entirely -- **BCM and OJA** don't qualify.
5 rules x 3 eta -- 892 cells total.

Fit via a JIT-compiled JAX Baum-Welch EM (converges to relative log-likelihood change
`< 1e-6`), verified against a reference numpy implementation to float64 precision.


In [ ]:
def has_tau(rule):
    shape, _ = npz_shape(FOLDER + f'det_{rule}_{SIZE}.npz')
    return len(shape) == 6

REWARD_THRESH = 0.7
N_ETA_PER_RULE = 3
MAX_ITER = 300     # safety cap; while_loop normally converges before this
TOL = 1e-6
RUN_SWEEP = False  # set True to force recompute

def build_manifest():
    """(rule, eta, best tau or None, [(a,w),...]) for every cell with reward>REWARD_THRESH
    at that eta. Rules with no qualifying eta are skipped entirely."""
    manifest = []
    for rule in RULES:
        reward = np.load(FOLDER + f'det_{rule}_{SIZE}.npy')
        tau = has_tau(rule)
        if tau:
            frac_by_tau = (reward > REWARD_THRESH).mean(axis=(0, 2, 3))
            t_idx = int(np.argmax(frac_by_tau))
            r2 = reward[:, t_idx]
        else:
            t_idx = None
            r2 = reward
        frac_by_eta = (r2 > REWARD_THRESH).mean(axis=(1, 2))
        eta_order = np.argsort(-frac_by_eta)
        eta_sel = [int(e) for e in eta_order if frac_by_eta[e] > 0.02][:N_ETA_PER_RULE]
        if not eta_sel:
            continue
        for e in eta_sel:
            aw = np.argwhere(r2[e] > REWARD_THRESH)
            manifest.append(dict(rule=rule, eta=e, tau=t_idx,
                                  cells=[(int(a), int(w)) for a, w in aw]))
    return manifest


def process_unit(u):
    reward = np.load(FOLDER + f"det_{u['rule']}_{SIZE}.npy")
    tau = has_tau(u['rule'])
    rvals = ([reward[u['eta'], u['tau'], a, w] for a, w in u['cells']] if u['tau'] is not None
              else [reward[u['eta'], a, w] for a, w in u['cells']])
    for i, arr in npz_leading_slices(FOLDER + f"det_{u['rule']}_{SIZE}.npz", indices=[u['eta']]):
        sl = arr[u['tau']] if tau else arr   # (T,seeds,alpha,beta)
    obs = np.stack([sl[:, :, a, w] for a, w in u['cells']], axis=1).astype(np.float64)

    pi0, A0, p0 = init_params(obs)
    obs_j = jnp.asarray(obs)
    pi_j, A_j, p_j, ll_j, n_iter_run = fit_hmm2(
        obs_j, MAX_ITER, jnp.asarray(pi0), jnp.asarray(A0), jnp.asarray(p0), TOL)
    pi_f, A_f, p_f = relabel_state1_high(np.array(pi_j), np.array(A_j), np.array(p_j))
    loglik = np.array(ll_j)
    if int(n_iter_run) >= MAX_ITER:
        print(f"  WARNING: {u['rule']} eta={u['eta']} hit MAX_ITER={MAX_ITER} without converging")

    nl = null_loglik(obs)
    delta2ll = 2 * (loglik - nl)
    vpath = np.array(viterbi(obs_j, jnp.asarray(pi_f), jnp.asarray(A_f), jnp.asarray(p_f)))
    ds = dwell_stats(vpath)
    return dict(rule=u['rule'], eta=u['eta'], tau=u['tau'], cells=u['cells'],
                reward=np.array(rvals), p=p_f, A=A_f, pi=pi_f,
                delta2ll=delta2ll, med_switch=np.median(ds['n_switches'], axis=1),
                min_switch=ds['n_switches'].min(axis=1),
                mean_dwell0=ds['mean_dwell0'], mean_dwell1=ds['mean_dwell1'])


def consolidate_to_summary(results):
    rows = []
    for d in results:
        for c, (a, w) in enumerate(d['cells']):
            rows.append((d['rule'], d['eta'], -1 if d['tau'] is None else d['tau'], a, w,
                         d['reward'][c], d['p'][c, 0], d['p'][c, 1], d['delta2ll'][c],
                         d['med_switch'][c], d['min_switch'][c],
                         d['mean_dwell0'][c], d['mean_dwell1'][c]))
    dt = np.dtype([('rule', 'U4'), ('eta', 'i4'), ('tau', 'i4'), ('a', 'i4'), ('w', 'i4'),
                   ('reward', 'f8'), ('p0', 'f8'), ('p1', 'f8'), ('delta2ll', 'f8'),
                   ('med_switch', 'f8'), ('min_switch', 'f8'), ('dwell0', 'f8'), ('dwell1', 'f8')])
    return np.array(rows, dtype=dt)


summary_path = CACHE_DIR + 'det_small_summary.npy'
if RUN_SWEEP or not os.path.exists(summary_path):
    manifest = build_manifest()
    print(f"{len(manifest)} units, {sum(len(u['cells']) for u in manifest)} cells "
          f"({[r for r in RULES if not any(u['rule'] == r for u in manifest)]} excluded -- "
          f"no eta clears reward>{REWARD_THRESH})")
    results = [process_unit(u) for u in manifest]
    summary = consolidate_to_summary(results)
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.save(summary_path, summary)
else:
    summary = np.load(summary_path)

print(f"summary: {len(summary)} cells across {len(set(summary['rule']))} rules")


## Classification protocol

A cell is **discretely bistable** only if all four hold:

1. **Significant**: `2*deltaLL > 200` (empirical null ceiling ~-8.5).
2. **Separated**: `p1 - p0 > 0.25`.
3. **Switches**: median Viterbi state-changes per seed >= 3.
4. **Non-degenerate dwell**: mean dwell time in both states >= 1.5 trials.

Criteria 3-4 catch seed-multistable look-alikes (different seeds locked onto different
fixed points, no seed ever switches, but still bimodal to the likelihood test). Cells
passing (1)+(2) but failing (3) or (4) are labelled **"fixed"**; everything else is **"ns"**.


In [ ]:
SIG_LL, GAP, MIN_SWITCH, MIN_DWELL = 200, 0.25, 3, 1.5

def classify(row):
    sig = (row['delta2ll'] > SIG_LL) and (row['p1'] - row['p0'] > GAP)
    if not sig:
        return 'ns'
    dyn = (row['med_switch'] >= MIN_SWITCH and row['dwell0'] >= MIN_DWELL
           and row['dwell1'] >= MIN_DWELL)
    return 'DYN' if dyn else 'FIX'

labels = np.array([classify(r) for r in summary])


In [ ]:
dyn_mask = labels == 'DYN'
dyn = summary[dyn_mask]
print(f"n discretely-bistable cells: {len(dyn)}")


## One representative parameter set per rule: which cell switches most consistently?

Per rule, pick the discretely-bistable cell maximizing the **minimum** Viterbi switch count
across all 20 seeds -- not just the best-separated `(p0,p1)`, which can still be a poor
example if only some seeds actually recur. Maximizing the worst-case seed's switch count
ensures the chosen cell recurs in every seed, not just on average.


In [ ]:
best_per_rule = {}
print(f"{'rule':<5s} {'n_disc':>6s} {'min_switch':>11s} {'med_switch':>11s} {'(a,w)':>8s} {'p0':>6s} {'p1':>6s}")
for rule in RULE_ORDER:
    sub = dyn[dyn['rule'] == rule]
    if len(sub) == 0:
        continue
    best = sub[np.argmax(sub['min_switch'])]
    best_per_rule[rule] = best
    aw_str = f"({best['a']},{best['w']})"
    print(f"{rule:<5s} {len(sub):>6d} {best['min_switch']:>11.0f} {best['med_switch']:>11.0f} "
          f"{aw_str:>8s} {best['p0']:>6.3f} {best['p1']:>6.3f}")


In [ ]:
# Full-session (T=50000, 20 seeds) Viterbi decode per selected cell -- refits each cell
# alone (same fit_hmm2/viterbi, C=1) since the sweep above batches cells per eta.
RECOMPUTE_QUALIFYING = False
states_path = CACHE_DIR + 'qualifying_states.npz'
cells_path = CACHE_DIR + 'qualifying_cells.npy'

RULES5 = [r for r in RULE_ORDER if r in best_per_rule]

if RECOMPUTE_QUALIFYING or not (os.path.exists(states_path) and os.path.exists(cells_path)):
    state_cache, meta_rows = {}, []
    for rule, row in best_per_rule.items():
        eta, tau, a, w = int(row['eta']), int(row['tau']), int(row['a']), int(row['w'])
        tau_r = has_tau(rule)
        for i, arr in npz_leading_slices(FOLDER + f'det_{rule}_{SIZE}.npz', indices=[eta]):
            sl = arr[tau] if tau_r else arr    # (T,seeds,alpha,beta)
        obs = sl[:, :, a, w].astype(np.float64)[:, None, :]   # (T,1,S)
        pi0, A0, p0 = init_params(obs)
        pi_j, A_j, p_j, ll_j, n_iter_run = fit_hmm2(
            jnp.asarray(obs), MAX_ITER, jnp.asarray(pi0), jnp.asarray(A0), jnp.asarray(p0), TOL)
        pi_f, A_f, p_f = relabel_state1_high(np.array(pi_j), np.array(A_j), np.array(p_j))
        path = np.array(viterbi(jnp.asarray(obs), jnp.asarray(pi_f), jnp.asarray(A_f), jnp.asarray(p_f)))
        state_cache[rule + '_state'] = path[:, 0, :].astype(np.int8)
        meta_rows.append((rule, eta, tau, a, w, p_f[0, 0], p_f[0, 1]))
        print(f"{rule}: refit p={p_f[0].round(3)}  (selection used p={row['p0']:.3f},{row['p1']:.3f})")
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.savez_compressed(states_path, **state_cache)
    dt = np.dtype([('rule', 'U4'), ('eta', 'i4'), ('tau', 'i4'), ('a', 'i4'), ('w', 'i4'),
                   ('p0', 'f8'), ('p1', 'f8')])
    np.save(cells_path, np.array(meta_rows, dtype=dt))

qual_states = np.load(states_path)
qual_meta = np.load(cells_path)
print(f"\nloaded full-session states for: {RULES5}")


In [ ]:
# Rerun the 5 selected settings from scratch at T=5,000,000 (10 fresh seeds) with the live
# simulator, to check recurrence isn't an artifact of the shorter T=50,000 window.
from ssb.scripts.run import run_quick
from ssb.envs.bandits import bandit_det

T_LONG = 5_000_000
N_SEEDS_LONG = 10
RECOMPUTE_LONG = False
long_states_path = CACHE_DIR + 'five_million_step_states.npz'

if RECOMPUTE_LONG or not os.path.exists(long_states_path):
    long_state_cache = {}
    for rule, row in best_per_rule.items():
        eta_v = [0, 1e-5, 1e-4, 1e-3, 0.01, 0.1][int(row['eta'])]
        # NOTE: the sweep's tau grid is logspace(-3, -1, 4) = [1e-3, 4.64e-3, 0.0215, 0.1]. The values below
        # (10**-2.3 = 5.0e-3, 10**-1.6 = 0.0251) are what produced Supp. Fig. 6b, so LC and LPL were rerun at
        # tau = 0.0251 rather than their swept 0.0215 (stated in the Supp. Fig. 6 legend).
        tau_v = [1e-3, 10**-2.3, 10**-1.6, 0.1][int(row['tau'])] if int(row['tau']) >= 0 else 0.0
        alpha_v = float(np.logspace(-5, 0, 15)[int(row['a'])])
        beta_v = float(np.logspace(-5, 0, 15)[int(row['w'])])

        log = run_quick(rule, alpha_v, beta_v, eta_v, bandit_det, timesteps=T_LONG,
                         size='small', seed=7001, n_seeds=N_SEEDS_LONG, tau=tau_v, reward=1.0)
        obs = (np.array(log[0]) == 1).astype(np.float64)[:, None, :]   # (T,1,S)

        pi0, A0, p0 = init_params(obs)
        pi_j, A_j, p_j, ll_j, n_iter_run = fit_hmm2(
            jnp.asarray(obs), MAX_ITER, jnp.asarray(pi0), jnp.asarray(A0), jnp.asarray(p0), TOL)
        pi_f, A_f, p_f = relabel_state1_high(np.array(pi_j), np.array(A_j), np.array(p_j))
        path = np.array(viterbi(jnp.asarray(obs), jnp.asarray(pi_f), jnp.asarray(A_f), jnp.asarray(p_f)))
        long_state_cache[rule + '_state'] = path[:, 0, :].astype(np.int8)
        print(f"{rule}: p=[{p_f[0,0]:.3f},{p_f[0,1]:.3f}]  (T=50,000 fit was "
              f"[{row['p0']:.3f},{row['p1']:.3f}])  reward={obs.mean():.3f}")
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.savez_compressed(long_states_path, **long_state_cache)

long_states = np.load(long_states_path)
print(f"loaded T={T_LONG} states for: {[r for r in RULE_ORDER if r + '_state' in long_states.files]}")


## Supplementary figure: the selection process, step by step (`sf_explore_process.svg`)

**Top** -- fit+classify (every dumbbell = one DYN cell's `(p0,p1)`), then pick the
max-min-switch cell (bold, `*`).

**Bottom** -- does that choice hold at `T=5,000,000` (10 fresh seeds)? Cumulative
exploit-episode count per seed.


In [ ]:
rule_color = {'LC': '#8C7DAA', 'COV': '#B96470', 'HR': '#82B48C', 'HZ': '#E1AA50', 'PC': '#738CAA',
              'BCM': '#999999', 'OJA': '#bbbbbb'}

def bout_starts(path_1d):
    """True at trials where an exploit episode begins."""
    starts = np.zeros(len(path_1d), dtype=bool)
    starts[0] = path_1d[0] == 1
    starts[1:] = (path_1d[1:] == 1) & (path_1d[:-1] == 0)
    return starts

def compact_k(x, pos):
    if x == 0:
        return '0'
    return f'{x/1000:.1f}k'.replace('.0k', 'k')

GAP_ROWS = 8
N_PLOT_POINTS = 300
TOP_WIDTH_FRAC = 0.62

fig = plt.figure(figsize=(6.6, 6.3))
gs = fig.add_gridspec(2, 1, height_ratios=[2.0, 1.0], hspace=0.5)
gs_top = gs[0].subgridspec(1, 2, width_ratios=[TOP_WIDTH_FRAC, 1.0 - TOP_WIDTH_FRAC])
axT = fig.add_subplot(gs_top[0])
gs_bot = gs[1].subgridspec(1, len(RULES5), wspace=0.55)
axes_bot = [fig.add_subplot(gs_bot[i]) for i in range(len(RULES5))]

# TOP: fit+classify -> pick most consistent
sel_by_rule = {row['rule']: row for row in qual_meta}
order = [r for r in reversed(RULE_ORDER) if (dyn['rule'] == r).any()]

yy, yticks, yticklabs = 0, [], []
for r in order:
    m = dyn['rule'] == r
    idx = np.argsort(dyn['p0'][m])
    p0s, p1s = dyn['p0'][m][idx], dyn['p1'][m][idx]
    # match cells by (eta,tau,a,w), not fitted p0/p1 (batched vs single-cell refits differ slightly)
    aw = np.stack([dyn['eta'][m], dyn['tau'][m], dyn['a'][m], dyn['w'][m]], axis=1)[idx]
    sel_key = ((int(sel_by_rule[r]['eta']), int(sel_by_rule[r]['tau']),
                int(sel_by_rule[r]['a']), int(sel_by_rule[r]['w']))
               if r in sel_by_rule else None)
    for row_i, (p0v, p1v) in enumerate(zip(p0s, p1s)):
        is_sel = sel_key is not None and tuple(int(x) for x in aw[row_i]) == sel_key
        lw, alpha, ms = (1.8, 1.0, 3.4) if is_sel else (0.6, 0.32, 1.3)
        z = 3 if is_sel else 1
        axT.plot([p0v, p1v], [yy, yy], color=rule_color[r], lw=lw, alpha=alpha, zorder=z)
        axT.plot([p0v, p1v], [yy, yy], 'o', color=rule_color[r], ms=ms, alpha=alpha, zorder=z)
        yy += 1
    yticks.append(yy - m.sum() / 2 - 0.5); yticklabs.append(r + ('*' if r in sel_by_rule else ''))
    yy += GAP_ROWS

axT.set_xlim(-0.02, 1.02)
nature_ax(axT, xlabel='state probability p(rewarded arm)', nticks=4)
axT.set_yticks(yticks); axT.set_yticklabels(yticklabs, fontsize=8)
axT.set_title('Step 1: fit + classify  →  Step 2: keep max-min-\nswitch cell (bold, *)', fontsize=8.5)

# BOTTOM: recurrence at T=5,000,000
for ax, rule in zip(axes_bot, RULES5):
    state = long_states[rule + '_state']
    T, S = state.shape
    ds = max(1, T // N_PLOT_POINTS)
    t = np.arange(T)[::ds]
    cum = np.stack([np.cumsum(bout_starts(state[:, s])) for s in range(S)])
    for s in range(S):
        ax.plot(t, cum[s, ::ds], color=rule_color[rule], lw=0.5, alpha=0.4)
    med = np.median(cum, axis=0)
    ax.plot(t, med[::ds], color=rule_color[rule], lw=1.6)
    nature_ax(ax, xlabel='trial', nticks=3)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(compact_k))
    ax.set_xlim(0, T); ax.set_ylim(bottom=0)
    ax.set_xticks([0, T // 2, T]); ax.set_xticklabels(['0', '2.5M', '5M'], fontsize=6)
    ax.set_title(f'{rule}\n{int(med[-1])} bouts', fontsize=7.5)
axes_bot[0].set_ylabel('cumulative\nexploit episodes', fontsize=8)

plt.savefig('./svgs/sf_explore_process.svg', dpi=150, bbox_inches='tight')
plt.show()


## Final figure: state recurrence (`f5_explore.svg`)

5 selected cells (T=50,000, 20 seeds). **Left** -- Viterbi state, trials 0-5,000 (20-trial
bins); `#6d6e71`=exploit, white=explore; seeds sorted by occupancy. **Right** -- median
cumulative exploit-episode count, normalized per rule; dashed diagonal = constant rate.


In [ ]:
from matplotlib.patches import Patch
import matplotlib.colors as mcolors

EXPLOIT_COLOR = '#6d6e71'
EXPLORE_COLOR = '#ffffff'
cmap_state = mcolors.LinearSegmentedColormap.from_list('explexp', [EXPLORE_COLOR, EXPLOIT_COLOR])

ZOOM_W0, ZOOM_W1 = 0, 5000
BLOCK_GAP = 6
BIN_ZOOM = 20

fig, (axM, axR) = plt.subplots(1, 2, figsize=(6.2, 3.3))

# LEFT: zoomed raster
S = 20
yy, yticks2, yticklabs2 = 0, [], []
for rule in reversed(RULES5):
    state_full = qual_states[rule + '_state'].astype(float)
    T = state_full.shape[0]
    occ1_full = state_full.mean(axis=0)
    order_s = np.argsort(occ1_full)
    seg = state_full[ZOOM_W0:ZOOM_W1]
    nb = (ZOOM_W1 - ZOOM_W0) // BIN_ZOOM
    binned = seg[:nb * BIN_ZOOM].reshape(nb, BIN_ZOOM, S).mean(axis=1).T
    axM.imshow(binned[order_s], aspect='auto', cmap=cmap_state, vmin=0, vmax=1,
               interpolation='nearest', extent=[0, ZOOM_W1 - ZOOM_W0, yy, yy + S], origin='lower')
    yticks2.append(yy + S / 2); yticklabs2.append(rule)
    yy += S + BLOCK_GAP

axM.set_xlim(0, ZOOM_W1 - ZOOM_W0); axM.set_ylim(0, yy - BLOCK_GAP)
axM.set_xticks([0, ZOOM_W1 - ZOOM_W0]); axM.set_xticklabels([str(ZOOM_W0), str(ZOOM_W1)], fontsize=6)
axM.set_xlabel('trial', fontsize=8)
axM.set_yticks(yticks2); axM.set_yticklabels(yticklabs2, fontsize=7.5)
axM.tick_params(length=0)
for sp in axM.spines.values(): sp.set_visible(False)
axM.set_title(f'zoomed\n(trials {ZOOM_W0}-{ZOOM_W1})', fontsize=8)
legend_handles = [Patch(facecolor=EXPLORE_COLOR, edgecolor='#999', linewidth=0.4, label='explore'),
                  Patch(facecolor=EXPLOIT_COLOR, edgecolor='#999', linewidth=0.4, label='exploit')]
axM.legend(handles=legend_handles, fontsize=6.5, frameon=False, loc='upper center',
           ncol=2, bbox_to_anchor=(0.5, 1.22))

# RIGHT: recurrence-rate summary
N_PLOT_POINTS = 200
axR.plot([0, 1], [0, 1], transform=axR.transAxes, color='#bbb', lw=0.8, ls='--', zorder=0)
for rule in RULES5:
    state = qual_states[rule + '_state']
    Tb, Sb = state.shape
    downsample = max(1, Tb // N_PLOT_POINTS)
    cum = np.stack([np.cumsum(bout_starts(state[:, s])) for s in range(Sb)]).astype(float)
    cum_frac = cum / cum[:, -1:]
    med = np.median(cum_frac, axis=0)
    lo = np.percentile(cum_frac, 25, axis=0)
    hi = np.percentile(cum_frac, 75, axis=0)
    t = np.arange(Tb)[::downsample]
    axR.fill_between(t, lo[::downsample], hi[::downsample], color=rule_color[rule], alpha=0.15, linewidth=0)
    total = int(np.median(cum[:, -1]))
    axR.plot(t, med[::downsample], color=rule_color[rule], lw=1.4, label=f'{rule} ({total})')

axR.set_xlim(0, Tb); axR.set_ylim(0, 1)
nature_ax(axR, xlabel='trial', ylabel='cumulative fraction of total\nexploit episodes (median)', nticks=4)
axR.set_xticks([0, Tb // 2, Tb]); axR.set_xticklabels(['0', '25000', '50000'], fontsize=6)
axR.legend(fontsize=6, frameon=False, loc='lower right', title='rule (median total)', title_fontsize=6)
axR.set_title('recurrence rate is steady\n(dashed = constant rate)', fontsize=8)

plt.tight_layout()
plt.savefig('./svgs/f5_explore.svg', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Same "recurrence rate is steady" panel as f5_explore.svg (right), but on the
# T=5,000,000 rerun (10 fresh seeds, `long_states` from the cell above).
N_PLOT_POINTS = 200

fig, axR5 = plt.subplots(1, 1, figsize=(3.1, 3.3))
axR5.plot([0, 1], [0, 1], transform=axR5.transAxes, color='#bbb', lw=0.8, ls='--', zorder=0)
for rule in RULES5:
    state = long_states[rule + '_state']
    Tb, Sb = state.shape
    downsample = max(1, Tb // N_PLOT_POINTS)
    # downsample right after cumsum (identical plotted values, avoids ~GBs of float arrays)
    cum = np.stack([np.cumsum(bout_starts(state[:, s]))[::downsample] for s in range(Sb)]).astype(float)
    final = np.array([bout_starts(state[:, s]).sum() for s in range(Sb)], dtype=float)
    cum_frac = cum / final[:, None]
    med = np.median(cum_frac, axis=0)
    lo = np.percentile(cum_frac, 25, axis=0)
    hi = np.percentile(cum_frac, 75, axis=0)
    t = np.arange(Tb)[::downsample]
    axR5.fill_between(t, lo, hi, color=rule_color[rule], alpha=0.15, linewidth=0)
    total = int(np.median(final))
    axR5.plot(t, med, color=rule_color[rule], lw=1.4, label=f'{rule} ({total})')

axR5.set_xlim(0, Tb); axR5.set_ylim(0, 1)
nature_ax(axR5, xlabel='trial', ylabel='cumulative fraction of total\nexploit episodes (median)', nticks=4)
axR5.set_xticks([0, Tb // 2, Tb]); axR5.set_xticklabels(['0', '2.5M', '5M'], fontsize=6)
axR5.legend(fontsize=6, frameon=False, loc='lower right', title='rule (median total)', title_fontsize=6)
axR5.set_title('recurrence rate is steady\n(dashed = constant rate)', fontsize=8)

plt.tight_layout()
plt.savefig('./svgs/f5_explore_recurrence_5M.svg', dpi=150, bbox_inches='tight')
plt.show()


## PC run-length distributions across feedback timing x reward structure (`f5_env_grid_PC.svg`)

On a **baited** ITI bandit: does PC's run-length distribution (power law vs exponential)
depend on feedback timing and reward structure? Uses `bandit_bait_iti` and the CSN
power-law fit (`ssb.analysis.powerlaw_fit`) -- unrelated to the HMM analysis above,
self-contained.

**Parameters:** `eta_a=0.01, alpha=0.0373, beta=0.0072` -- the operating point
`f3_continual/f3_forgetting.ipynb` uses for its retention analysis (not `best_params`,
which is too static here). `init_scale=0.25`, `clip=3.0`, `seek_thresh=0.5` carried over
unchanged from the reference figure. ITI rows 0 and 2 reproduce the same diagonal staircase:

| | P=1.00/0.00 | P=0.45/0.15 | P=0.05/0.05 |
|---|---|---|---|
| **ITI=0** | power law (a=1.46) | power law (a=1.49) | not significant |
| **ITI=2** | power law (a=1.52) | exponential (a=1.59) | exponential (a=1.63) |

(ITI=0/deterministic is right-censored: >=1 seed never switches in the 20,000-trial window.)


In [ ]:
import time
# bandit_bait_iti has an untyped float literal that promotes to float64 under this
# notebook's x64 mode; run this section with x64 off, restore after.
jax.config.update('jax_enable_x64', False)
from ssb.models import ssb as ssb_core, rules
from ssb.utils.batch_inits import make_default_hps, batchify_hps, init_params_batch
from ssb.envs.bandits import bandit_bait_iti, make_iti_state, make_iti_env_par
from ssb.analysis.powerlaw_fit import action_lengths, fit_and_compare

# PC operating point from f3_continual/f3_forgetting.ipynb's retention analysis.
ETA_A, ALPHA, BETA = 0.01, 0.0372759372, 0.0071968567
INIT_SCALE = 0.25
CLIP = 3.0
SEEK_THRESH = 0.5
ENV_SEEDS, ENV_TRIALS, ENV_BURN = 24, 120_000, 4_000  # trials raised from 20k to uncensor ITI=0/deterministic
ENV_ROWS = [(0, 'ITI=0'), (2, 'ITI=2')]
ENV_COLS = [(1.00, 0.00), (0.45, 0.15), (0.05, 0.05)]
BAR_COLOR = '#6d6e71'

print(f"PC params: eta_a={ETA_A}, alpha={ALPHA:.4g}, beta={BETA:.4g}")
print(f"init_scale={INIT_SCALE}, clip={CLIP}, seek_thresh={SEEK_THRESH}, "
      f"seeds={ENV_SEEDS}, trials={ENV_TRIALS}, burn={ENV_BURN}")


def _run_iti(iti, seed, n_trials, probs, alpha=ALPHA, beta=BETA, eta_a=ETA_A,
             init_scale=INIT_SCALE, burn_trials=ENV_BURN, clip=CLIP):
    """One PC-rule run on a baited ITI bandit. Returns choices on real bandit steps
    only (ITI null steps dropped); arm 1 = good arm."""
    iti = int(iti)
    sample = int(n_trials * (iti + 1)) + (iti + 1)
    burn   = int(burn_trials * (iti + 1))
    hps = make_default_hps(seed)
    hps['init_scale'] = jnp.array([init_scale])
    hps['alpha'] = jnp.array([alpha]); hps['beta'] = jnp.array([beta])
    hps['eta_a'] = jnp.array([eta_a])
    hb = batchify_hps(hps, 1)
    uw, us = rules.rule2fns('PC')
    UA = jax.vmap(ssb_core.update_acts); UW = jax.vmap(uw); US = jax.vmap(us)
    NO = jax.vmap(ssb_core.act_noise);   EX = jax.vmap(ssb_core.expand_stimulus)
    env = jax.vmap(bandit_bait_iti, in_axes=(0, 0, 0, 0))
    acts, states, weights, tern, key = init_params_batch(hb)
    # bandit_bait_iti's state is hardcoded float32; force explicitly to avoid a scan-carry
    # dtype mismatch under jax_enable_x64.
    env_state = jnp.asarray(make_iti_state(iti), dtype=jnp.float32)[None]
    env_par   = jnp.asarray(make_iti_env_par(probs[0], probs[1], iti), dtype=jnp.float32)[None]

    @jax.jit
    def step(c, _):
        acts, states, weights, es, key = c
        is_bandit = (es[:, 2] == 0)
        rb, action, nes, key = env(es, acts[:, -2:], env_par, key)
        stim = EX(rb, acts)
        acts = UA(stim, acts, states, weights, tern, hb)
        acts, key = NO(acts, key, hb); acts = jnp.clip(acts, 0., clip)
        weights = UW(stim, acts, states, weights, tern, hb)
        states = US(acts, states, hb)
        return (acts, states, weights, nes, key), (action[0], is_bandit[0])

    carry = (acts, states, weights, env_state, key)
    carry, _ = jax.lax.scan(step, carry, None, length=burn)
    _, out = jax.lax.scan(step, carry, None, length=sample)
    a = np.asarray(out[0]).ravel().astype(int)
    m = np.asarray(out[1]).ravel().astype(bool)
    return a[m]


def env_collect(pa, pb, iti, seeds=ENV_SEEDS, n_trials=ENV_TRIALS, burn=ENV_BURN):
    """Pool run lengths over seeking seeds (asymmetric arms) or all seeds (symmetric)."""
    symmetric = (pa == pb)
    f1, pooled, pooled_all = [], [], []
    for s in range(seeds):
        ch = _run_iti(iti, s, n_trials, probs=(pb, pa), burn_trials=burn)
        m1 = float(ch.mean()); f1.append(m1)
        rl = action_lengths(ch); pooled_all.append(rl)
        if m1 > SEEK_THRESH: pooled.append(rl)
    runs = (np.concatenate(pooled_all) if symmetric
            else (np.concatenate(pooled) if pooled else np.array([], int)))
    return dict(runs=runs, meanf1=float(np.mean(f1)))


# cache: 6 cells x 24 seeds x 120k trials is slow -- skip on repeat runs.
RECOMPUTE_ENV_GRID = False   # also delete the cache if operating-point params above change
env_cache_path = CACHE_DIR + 'f5_env_grid_PC.npy'

def _print_env_row(iti, pa, pb, c):
    mx = int(c['runs'].max()) if c['runs'].size else 0
    cens = ' [CENSORED]' if mx >= ENV_TRIALS else ''
    print(f"ITI={iti} a={pa:.2f} b={pb:.2f} | seek {100*c['meanf1']:4.0f}% | "
          f"nruns={c['runs'].size:6d} maxrun={mx:5d}{cens} | "
          + (f"{c['fit'].favored:11s} a={c['fit'].alpha:.2f} "
             f"LR={c['fit'].loglik_ratio:+.2f} p={c['fit'].loglik_ratio_p:.4f}" if c['fit'] else 'NA'))

t0 = time.time()
if RECOMPUTE_ENV_GRID or not os.path.exists(env_cache_path):
    env = {}
    for iti, _ in ENV_ROWS:
        for pa, pb in ENV_COLS:
            c = env_collect(pa, pb, iti)
            c['fit'] = fit_and_compare(c['runs'], xmin=1) if c['runs'].size >= 2 else None
            env[(iti, pa, pb)] = c
            _print_env_row(iti, pa, pb, c)
    print('total', round(time.time() - t0, 1), 's (computed fresh)')
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.save(env_cache_path, env, allow_pickle=True)
    print(f'cached to {env_cache_path}')
else:
    env = np.load(env_cache_path, allow_pickle=True).item()
    for iti, _ in ENV_ROWS:
        for pa, pb in ENV_COLS:
            _print_env_row(iti, pa, pb, env[(iti, pa, pb)])
    print(f'loaded cached env grid ({round(time.time() - t0, 1)}s) -- set RECOMPUTE_ENV_GRID=True to refresh')

# draw the 2x3 grid (feedback timing x reward structure)
SHARED_XMAX = 10_000   # capped below the biggest cell's true max so every panel shares one honest axis
bins = np.logspace(0, np.log10(SHARED_XMAX + 1), 20)

def _p_str(pval):
    stars = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    pstr = 'p<0.001' if pval < 0.001 else f'p={pval:.3f}'
    return f'{pstr} {stars}'.rstrip()

fig, axs = plt.subplots(len(ENV_ROWS), len(ENV_COLS), figsize=(6.6, 3.9), sharex=True, sharey=True)
fig.subplots_adjust(left=0.15, right=0.86, top=0.80, bottom=0.14, wspace=0.42, hspace=0.30)
for ri, (iti, rowtitle) in enumerate(ENV_ROWS):
    for ci, (pa, pb) in enumerate(ENV_COLS):
        ax = axs[ri, ci]; c = env[(iti, pa, pb)]; fit = c['fit']
        d, _ = np.histogram(c['runs'], bins=bins, density=True); m = d > 0
        ax.bar(bins[:-1][m], d[m], width=np.diff(bins)[m], align='edge', color=BAR_COLOR, edgecolor='none')
        ax.set_xscale('log'); ax.set_yscale('log')
        for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
        for sp in ('left', 'bottom'): ax.spines[sp].set_linewidth(0.25)
        for axis in (ax.xaxis, ax.yaxis):
            axis.set_major_locator(mticker.LogLocator(base=10, numticks=4))
            axis.set_major_formatter(mticker.LogFormatterMathtext(base=10))
            axis.set_minor_locator(mticker.NullLocator())
        ax.tick_params(which='major', labelsize=6, width=0.25, length=2.5)
        if fit is not None and fit.favored == 'power_law':
            ax.axvline(fit.xmin, color='#555555', lw=0.5, ls=':')
            ax.text(0.95, 0.95, 'power law', transform=ax.transAxes,
                    ha='right', va='top', fontsize=6.5, color='#2e7d32')
            ax.text(0.95, 0.86, _p_str(fit.loglik_ratio_p), transform=ax.transAxes,
                    ha='right', va='top', fontsize=5.5, color='#2e7d32')
        elif fit is not None and fit.favored == 'exponential':
            ax.text(0.62, 0.68, 'exp.', transform=ax.transAxes, ha='center', va='center',
                    fontsize=11, color='#b8b8b8', fontweight='bold')
            ax.text(0.62, 0.54, _p_str(fit.loglik_ratio_p), transform=ax.transAxes,
                    ha='center', va='center', fontsize=5.5, color='#b8b8b8')
        elif fit is not None:
            ax.text(0.62, 0.68, 'n.s.', transform=ax.transAxes, ha='center', va='center',
                    fontsize=11, color='#b8b8b8', fontweight='bold')
            ax.text(0.62, 0.54, _p_str(fit.loglik_ratio_p), transform=ax.transAxes,
                    ha='center', va='center', fontsize=5.5, color='#b8b8b8')
        ax.text(1.04, 0.5, f"seek {100*c['meanf1']:.0f}%", transform=ax.transAxes, rotation=270,
                ha='left', va='center', fontsize=5.5, color='#999999')
for ci in range(len(ENV_COLS)): axs[-1, ci].set_xlabel('run length L (trials)', fontsize=8)
for ri in range(len(ENV_ROWS)): axs[ri, 0].set_ylabel('PDF', fontsize=8, labelpad=1)
fig.canvas.draw()
for ci, (pa, pb) in enumerate(ENV_COLS):
    p = axs[0, ci].get_position()
    fig.text((p.x0 + p.x1) / 2, 0.845, f'P(s=1|A=a) = {pa:g}\nP(s=1|A=b) = {pb:g}',
              ha='center', va='bottom', fontsize=7.5)
for ri, (iti, rowtitle) in enumerate(ENV_ROWS):
    p = axs[ri, 0].get_position()
    fig.text(0.012, (p.y0 + p.y1) / 2, rowtitle, ha='left', va='center', fontsize=8.5, rotation=90)
fig.suptitle("PC foraging (continual-learning operating point): environment reshapes "
             "run-length distributions", fontsize=9, y=0.99, x=0.5)
fig.text(0.5, 0.005, '* p<0.05   ** p<0.01   *** p<0.001  (Vuong LR test, power law vs exponential)',
         ha='center', va='bottom', fontsize=6, color='#666666')
plt.savefig('./svgs/f5_env_grid_PC.svg', dpi=300, bbox_inches='tight')
plt.show()
print('saved ./svgs/f5_env_grid_PC.svg')

jax.config.update('jax_enable_x64', True)


## ATTRACTORS. Example choice traces (texture of the two regimes)

Same network, ITI=0 vs a spaced ITI. Continuous: long heavy-tailed dwells punctuated by
bursts. Spaced: switches on a roughly constant hazard. Each panel shows the realized
fraction of trials on the good arm.


In [ ]:
# ATTRACTORS setup: reuses the PC operating point from the run-length section above
# (ETA_A/ALPHA/BETA/INIT_SCALE/CLIP/SEEK_THRESH/ENV_SEEDS/ENV_TRIALS/ENV_BURN), restricted
# to ITIS=[0,2] (ENV_ROWS). Reward structure fixed at (0.45,0.15) -- the column that splits
# power law (ITI=0) / exponential (ITI=2); the deterministic (1,0) column is power law at
# both ITIs.
jax.config.update('jax_enable_x64', False)

GOOD_P, BAD_P = 0.45, 0.15
PROBS = (BAD_P, GOOD_P)
ITIS  = [iti for iti, _ in ENV_ROWS]
SEEDS = list(range(ENV_SEEDS))
N_TRIALS = ENV_TRIALS
EXPLOIT_CUTOFF = 50   # run length >= this -> "long dwell" (exploit)

C0 = '#967fb7'   # continuous / ITI=0 (purple -- same as the cloud/branch color below)
C1 = '#f4bc67'   # spaced / ITI>0 (gold -- same as the cloud color below)

def perstep_runlen(choices):
    """Length of the run each timestep belongs to."""
    rl = action_lengths(np.asarray(choices))
    return np.repeat(rl, rl) if rl.size else np.array([], int)

def _run(iti, seed, n_trials, want_state=False, probs=PROBS,
         alpha=ALPHA, beta=BETA, eta_a=ETA_A, init_scale=INIT_SCALE,
         burn_trials=ENV_BURN, clip=CLIP):
    """One PC-rule run on the baited ITI bandit. want_state=True also returns x_in,
    dW=w2-w1 (line-attractor axis), dX=x2-x1 (motor difference) on the bandit steps."""
    iti = int(iti)
    sample = int(n_trials * (iti + 1)) + (iti + 1)
    burn   = int(burn_trials * (iti + 1))
    hps = make_default_hps(seed)
    hps['init_scale'] = jnp.array([init_scale])
    hps['alpha'] = jnp.array([alpha]); hps['beta'] = jnp.array([beta])
    hps['eta_a'] = jnp.array([eta_a])
    hb = batchify_hps(hps, 1)
    uw, us = rules.rule2fns('PC')
    UA = jax.vmap(ssb_core.update_acts); UW = jax.vmap(uw); US = jax.vmap(us)
    NO = jax.vmap(ssb_core.act_noise);   EX = jax.vmap(ssb_core.expand_stimulus)
    env = jax.vmap(bandit_bait_iti, in_axes=(0, 0, 0, 0))
    acts, states, weights, tern, key = init_params_batch(hb)
    env_state = jnp.asarray(make_iti_state(iti), dtype=jnp.float32)[None]
    env_par   = jnp.asarray(make_iti_env_par(probs[0], probs[1], iti), dtype=jnp.float32)[None]

    @jax.jit
    def step(c, _):
        acts, states, weights, es, key = c
        is_bandit = (es[:, 2] == 0)
        rb, action, nes, key = env(es, acts[:, -2:], env_par, key)
        stim = EX(rb, acts)
        acts = UA(stim, acts, states, weights, tern, hb)
        acts, key = NO(acts, key, hb); acts = jnp.clip(acts, 0., clip)
        weights = UW(stim, acts, states, weights, tern, hb)
        states = US(acts, states, hb)
        dW = weights[0, 0, -1] - weights[0, 0, -2]
        dX = acts[0, -1] - acts[0, -2]
        return (acts, states, weights, nes, key), (action[0], is_bandit[0], acts[0, 0], dW, dX)

    carry = (acts, states, weights, env_state, key)
    carry, _ = jax.lax.scan(step, carry, None, length=burn)
    _, out = jax.lax.scan(step, carry, None, length=sample)
    a = np.asarray(out[0]).ravel().astype(int)
    m = np.asarray(out[1]).ravel().astype(bool)
    if want_state:
        xin = np.asarray(out[2]).ravel()[m]
        dW  = np.asarray(out[3]).ravel()[m]
        dX  = np.asarray(out[4]).ravel()[m]
        return a[m], xin, dW, dX
    return a[m]

def collect_state(iti, seeds, n_trials, max_seeds=6):
    """Pool attractor coords over up to max_seeds reward-seeking seeds at one ITI."""
    XIN, DW, DX, EXM = [], [], [], []
    used = 0
    for s in seeds:
        a, xin, dW, dX = _run(iti, s, n_trials, want_state=True)
        if a.mean() > SEEK_THRESH:
            XIN.append(xin); DW.append(dW); DX.append(dX)
            EXM.append(perstep_runlen(a) >= EXPLOIT_CUTOFF)
            used += 1
            if used >= max_seeds: break
    if not XIN:
        return None
    return dict(xin=np.concatenate(XIN), dW=np.concatenate(DW),
                dX=np.concatenate(DX), exploit=np.concatenate(EXM))

print(f"ATTRACTORS setup: eta_a={ETA_A}, alpha={ALPHA:.4g}, beta={BETA:.4g}, "
      f"probs={PROBS}, ITIS={ITIS}, {len(SEEDS)} seeds x {N_TRIALS} trials")


In [ ]:
## 3 (traces). Example traces: line- vs point-attractor texture
# CONTINUOUS (ITI=0): x2-x1 wanders far along the attractor's flat direction -- long high
# plateaus punctuated by dips (explore bouts). SPACED (ITI>0): weights reset each trial,
# x2-x1 stays in a tight band, switching on a roughly constant hazard.
#
# ATTR_ETA_A=0.002: lower than the shared ETA_A=0.01 (tuned for run-length statistics, not
# cloud shape) -- at 0.01 every seed's cloud is dominated by activity jitter regardless of
# ITI. Swept eta_a x 15 seeds at both ITI=0 and ITI=2: 0.002 is where ITI=0 clouds elongate
# along dW (the line-attractor axis) while ITI=2 clouds stay confined to a much smaller dW
# range -- ITI, not eta_a, drives the line-vs-blob contrast. Same eta_a for both regimes.
ATTR_ETA_A = 0.002
#
# CONT_SEED=6: highest cloud elongation of 15 ITI=0 seeds swept. Over the first TRACE_TRIALS
# (32,500) trials it chooses the good arm 97% of the time (587 runs); over the full 360,000-trial
# run used for Fig. 4f,h it is 76% (the number in the paper). Power law confirmed. Its opening dwell runs ~22,900 trials -- flat, no exploration
# visible. Right after it, the run lengths are [761,1,525,1,835,1,1,1,1553,6,585,...]:
# several long (300-1500 trial) dwells separated by brief (1-6 trial) dips -- long
# heavy-tailed dwells punctuated by bursts, the actual claim this panel makes. Widened
# WIN so several of these fit in one window rather than showing just the first 1-2 dips.
# SPACED_SEED=8: tightest of 15 ITI=2 seeds swept (dW range ~0.012 vs CONT's 0.18-0.25);
# 64% seeking, exponential confirmed.
CONT_ITI,   CONT_SEED,   CONT_WIN_START   = 0, 6, 32000
SPACED_ITI, SPACED_SEED, SPACED_WIN_START = 2, 8, 0
WIN     = 500
SMOOTH_K = 11
TRACE_TRIALS = max(6000, CONT_WIN_START + WIN, SPACED_WIN_START + WIN)

def _smooth(x, k):
    return x if k <= 1 else np.convolve(x, np.ones(k) / k, mode='same')

traces = {}
for tag, iti, seed, st in (('cont', CONT_ITI, CONT_SEED, CONT_WIN_START),
                           ('spaced', SPACED_ITI, SPACED_SEED, SPACED_WIN_START)):
    ch, _, _, dX = _run(iti, seed, TRACE_TRIALS, want_state=True, eta_a=ATTR_ETA_A)
    traces[tag] = dict(iti=iti, seed=seed, st=st, chose_good=float(ch.mean()),
                       a=ch[st:st + WIN], dX=_smooth(dX, SMOOTH_K)[st:st + WIN])

ymax = 1.15 * max(np.abs(traces[t]['dX']).max() for t in traces)

fig, axs = plt.subplots(4, 1, figsize=(1.5, 3.4), sharex=True,
                        gridspec_kw={'height_ratios': [2, 1, 2, 1], 'hspace': 0.22})
panels = [('cont', C0, 'continuous'), ('spaced', C1, 'spaced')]
for r, (tag, col, label) in enumerate(panels):
    d = traces[tag]; ax_traj, ax_ch = axs[2 * r], axs[2 * r + 1]
    t = np.arange(WIN)
    ax_traj.axhline(0, color='#cccccc', lw=0.4, ls=':')
    ax_traj.plot(t, d['dX'], color='#333333', lw=0.6)
    ax_traj.set_ylim(-ymax, ymax)
    ax_traj.set_ylabel('$x_2-x_1$', fontsize=7)
    ax_traj.set_title(f"{label} (ITI={d['iti']}, seed {d['seed']})", fontsize=7.5, pad=2)
    ax_ch.plot(t, d['a'], color=col, lw=0.4, drawstyle='steps-mid')
    ax_ch.set_yticks([0, 1]); ax_ch.set_yticklabels(['dead', 'good'], fontsize=7)
    ax_ch.text(0.995, 0.5, f"chose good arm: {d['chose_good']:.0%}", transform=ax_ch.transAxes,
               ha='right', va='center', fontsize=6, color='#555555')
    for ax in (ax_traj, ax_ch):
        for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
        for sp in ('left', 'bottom'): ax.spines[sp].set_linewidth(0.25)
        ax.tick_params(labelsize=7, width=0.25)
axs[-1].set_xlabel('bandit trial', fontsize=9)
fig.suptitle(r'$x_2-x_1$ wanders far (line attractor) when continuous; stays tight (point-like) when spaced',
             fontsize=8.5, y=1.02)
plt.tight_layout()
plt.savefig('./svgs/f5_traces.svg')
plt.show()


## 3b. Attractor clouds for the two example runs above

Same geometry as section 4, but for the exact runs plotted above (single seed per panel,
one-to-one with each trace). Continuous spreads along $w_2-w_1$ (heavy-tailed); spaced
collapses onto a tight band. Continuous cloud in purple, spaced in gold.


In [ ]:
# Single-seed version of section 4's draw_attractor2d, for the exact (ITI,seed) picked
# above -- both regimes at the same ATTR_ETA_A. Colors: C0/C1 (same as traces/histogram).
ATT_TRIALS_TR = N_TRIALS * 3

def _state_single(iti, seed, n_trials, eta_a=ATTR_ETA_A):
    """Attractor coords (x_in, dW=w2-w1, dX=x2-x1) + long-dwell mask for one run."""
    a, xin, dW, dX = _run(iti, seed, n_trials, want_state=True, eta_a=eta_a)
    return dict(a=a, xin=xin, dW=dW, dX=dX,
                exploit=perstep_runlen(a) >= EXPLOIT_CUTOFF,
                chose_good=float(a.mean()))

def _style2d(ax):
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'): ax.spines[sp].set_linewidth(0.25)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(3))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(3))
    ax.tick_params(labelsize=6, width=0.25, length=2.5, colors='#555555')
    ax.set_xlabel('$w_2-w_1$', fontsize=8); ax.set_ylabel('$x_2-x_1$', fontsize=8)

def draw_trace_attractors(n_show=3000, save=None):
    """One 2D attractor cloud per example run (continuous, spaced). Subsamples only
    the dense short-dwell core; every long-dwell ("exploit") point is kept, so the
    branch isn't thinned out by uniform subsampling like the dense core is."""
    runs = [('continuous', CONT_ITI, CONT_SEED, C0),
            ('spaced',     SPACED_ITI, SPACED_SEED, C1)]
    st = {tag: _state_single(iti, seed, ATT_TRIALS_TR) for tag, iti, seed, _ in runs}
    plt.close('all'); rng = np.random.default_rng(0)
    def lim(key):
        a = np.concatenate([st[t][key] for t, _, _, _ in runs])
        pad = 0.05 * ((a.max() - a.min()) or 1.0)
        return a.min() - pad, a.max() + pad
    xl, yl = lim('dW'), lim('dX')
    fig, axs = plt.subplots(1, 2, figsize=(1.9 * 2, 1.9), squeeze=False); axs = axs[0]
    for ax, (tag, iti, seed, col) in zip(axs, runs):
        d = st[tag]; ex = d['exploit']
        core = np.where(~ex)[0]
        if len(core) > n_show: core = rng.choice(core, n_show, replace=False)
        tail = np.where(ex)[0]
        ax.scatter(d['dW'][core], d['dX'][core], s=2, c=col,
                   alpha=0.20, linewidths=0, rasterized=True)
        ax.scatter(d['dW'][tail], d['dX'][tail], s=3, c=col,
                   alpha=0.55, linewidths=0, rasterized=True)
        ax.set_xlim(*xl); ax.set_ylim(*yl); ax.set_box_aspect(1); _style2d(ax)
        ax.set_title(f'{tag} (ITI={iti}, seed {seed})', fontsize=8, pad=2)
        ax.text(0.5, -0.34, f"chose good arm: {d['chose_good']:.0%}", transform=ax.transAxes,
                ha='center', va='top', fontsize=6, color='#555555')
    handles = [Patch(color=C0, label='continuous (ITI=0)'),
               Patch(color=C1, label=f'spaced (ITI={SPACED_ITI})')]
    fig.legend(handles=handles, fontsize=6, frameon=False, loc='lower center',
               ncol=2, bbox_to_anchor=(0.5, -0.12))
    fig.suptitle('Attractor clouds for the two example runs above', fontsize=8, y=1.04)
    plt.tight_layout()
    if save: plt.savefig(save, dpi=300, bbox_inches='tight')
    plt.show()

draw_trace_attractors(save='./svgs/f5_trace_attractors.svg')
print('saved ./svgs/f5_trace_attractors.svg')


## 3c. Run-length histograms for the two example runs above

Same two runs as the traces/clouds above (continuous ITI=0 seed 6, spaced ITI=2 seed 8):
pooled run-length distribution over the full `ATT_TRIALS_TR`-trial simulation, with the
same CSN power-law-vs-exponential test used in the PC run-length grid earlier.


In [ ]:
def draw_runlength_hist(save=None, nbins=20):
    """Run-length PDF (log-log) for the two example runs, with the CSN power-law-vs-
    exponential verdict per panel. Both panels share the same x/y axis range so the
    reach of one distribution's tail against the other's cutoff is a direct, honest
    visual comparison, not an artifact of separately-scaled axes."""
    runs = [('continuous', CONT_ITI, CONT_SEED, C0), ('spaced', SPACED_ITI, SPACED_SEED, C1)]
    rls = {}
    for tag, iti, seed, col in runs:
        d = _state_single(iti, seed, ATT_TRIALS_TR)
        rls[tag] = action_lengths(d['a'])
    xmax = max(rl.max() for rl in rls.values())
    bins = np.logspace(0, np.log10(xmax + 1), nbins)
    denses = {}
    for tag, _, _, _ in runs:
        d, _ = np.histogram(rls[tag], bins=bins, density=True)
        denses[tag] = d
    ymin = min(d[d > 0].min() for d in denses.values())
    ymax = max(d.max() for d in denses.values())

    fig, axs = plt.subplots(1, 2, figsize=(4.2, 2.0))
    for ax, (tag, iti, seed, col) in zip(axs, runs):
        rl = rls[tag]
        fit = fit_and_compare(rl, xmin=1)
        d = denses[tag]; m = d > 0
        ax.bar(bins[:-1][m], d[m], width=np.diff(bins)[m], align='edge', color=col, edgecolor='none')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlim(bins[0], bins[-1]); ax.set_ylim(ymin * 0.5, ymax * 2)
        for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
        for sp in ('left', 'bottom'): ax.spines[sp].set_linewidth(0.25)
        for axis in (ax.xaxis, ax.yaxis):
            axis.set_major_locator(mticker.LogLocator(base=10, numticks=4))
            axis.set_major_formatter(mticker.LogFormatterMathtext(base=10))
            axis.set_minor_locator(mticker.NullLocator())
        ax.tick_params(which='major', labelsize=6, width=0.25, length=2.5)
        verdict = {'power_law': 'power law', 'exponential': 'exponential', 'inconclusive': 'n.s.'}[fit.favored]
        vcol = '#2e7d32' if fit.favored == 'power_law' else '#888888'
        ax.text(0.95, 0.95, verdict, transform=ax.transAxes, ha='right', va='top', fontsize=7, color=vcol)
        ax.text(0.95, 0.83, f'α={fit.alpha:.2f}', transform=ax.transAxes, ha='right', va='top', fontsize=6, color=vcol)
        ax.text(0.95, 0.72, _p_str(fit.loglik_ratio_p), transform=ax.transAxes, ha='right', va='top', fontsize=6, color=vcol)
        ax.set_title(f'{tag} (ITI={iti}, seed {seed})', fontsize=8, pad=2)
        ax.set_xlabel('run length L (trials)', fontsize=7.5)
    axs[0].set_ylabel('PDF', fontsize=8)
    fig.suptitle('Run-length distributions for the two example runs above', fontsize=8.5, y=1.06)
    fig.text(0.5, -0.04, '* p<0.05   ** p<0.01   *** p<0.001  (Vuong LR test, power law vs exponential)',
             ha='center', va='top', fontsize=6, color='#666666')
    plt.tight_layout()
    if save: plt.savefig(save, dpi=300, bbox_inches='tight')
    plt.show()

draw_runlength_hist(save='./svgs/f5_example_runlengths.svg')
print('saved ./svgs/f5_example_runlengths.svg')
